# LangGraph 02 — Groq: Semantic Router với Structured Output

Demo xây hệ thống support có router dùng LLM để phân loại yêu cầu, sau đó LangGraph route sang node chuyên môn:

`START → classify → billing | technical | general | human → END`

In [2]:
import os
from getpass import getpass

def ensure_secret(name: str):
    if os.getenv(name):
        return
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = getpass(f"Nhập {name}: ")
    if not value:
        raise ValueError(f"Thiếu {name}")
    os.environ[name] = value

ensure_secret("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")
print("Model:", GROQ_MODEL)

Model: openai/gpt-oss-20b


## 1. Định nghĩa state và schema của router

Structured output biến quyết định tự do của LLM thành dữ liệu có schema. Confidence thấp sẽ được chuyển sang nhánh `human`, giúp workflow có điểm kiểm soát rõ ràng.

In [4]:
from typing import Literal, TypedDict
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langgraph.graph import END, START, StateGraph

Category = Literal["billing", "technical", "general", "human"]

class SupportState(TypedDict, total=False):
    # Code câu hỏi, loại, câu trả lời support, confidence (tin tưởng bao nhiêu % trong quyết định chọn agent xử lý report)
    question: str
    category: Category
    confidence: float # Độ tự tin khi chọn agent xử lý report
    rationale: str # Trình bày lý do tại sao chọn agent xử lý report
    answer: str
    

class RouteDecision(BaseModel):
    # Code
    category: Literal["billing", "technical", "general"] = Field(
        description="Nhóm phù hợp cho câu hỏi"
    )
    confidence: float = Field(ge=0,le=1)
    rationale: str = Field(description="Lý do chọn ngắn gọn bằng tiếng Việt")

In [5]:
# llm & router_llm
llm = ChatGroq(model=GROQ_MODEL, temperature=0)
router_llm = llm.with_structured_output(RouteDecision)

In [6]:
def classify(state: SupportState):
    decision = router_llm.invoke(
        "Phân loại yêu cầu support sau. "
        "billing: thanh toán/hóa đơn; technical: lỗi kỹ thuật; general: thông tin chung.\n"
        f"Yêu cầu: {state['question']}"
    )
    category: Category = decision.category if decision.confidence >= 0.55 else "human"
    return {
        "category": category,
        "confidence": decision.confidence,
        "rationale": decision.rationale,
    }

def billing_agent(state: SupportState):
    reply = llm.invoke(
        "Bạn là chuyên viên billing. Không bịa dữ liệu tài khoản; nếu thiếu dữ liệu hãy nói rõ. "
        f"Trả lời tiếng Việt cho: {state['question']}"
    )
    return {"answer": reply.content}

def technical_agent(state: SupportState):
    reply = llm.invoke(
        "Bạn là kỹ sư support. Đưa checklist chẩn đoán ngắn, an toàn, bằng tiếng Việt. "
        f"Yêu cầu: {state['question']}"
    )
    return {"answer": reply.content}

def general_agent(state: SupportState):
    reply = llm.invoke(
        "Bạn là nhân viên CSKH. Trả lời thân thiện, ngắn gọn bằng tiếng Việt. "
        f"Yêu cầu: {state['question']}"
    )
    return {"answer": reply.content}

def human_fallback(state: SupportState):
    return {
        "answer": "Yêu cầu chưa đủ rõ hoặc router thiếu tự tin. Đã chuyển cho nhân viên hỗ trợ."
    }

def route(state: SupportState) -> Category:
    return state["category"]

In [7]:
builder = StateGraph(SupportState)
builder.add_node("classify", classify)
builder.add_node("billing", billing_agent)
builder.add_node("technical", technical_agent)
builder.add_node("general", general_agent)
builder.add_node("human", human_fallback)
builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    route,
    {
        "billing": "billing",
        "technical": "technical",
        "general": "general",
        "human": "human",
    },
)
for node in ["billing", "technical", "general", "human"]:
    builder.add_edge(node, END)

support_graph = builder.compile()
print(support_graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify(classify)
	billing(billing)
	technical(technical)
	general(general)
	human(human)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify;
	classify -.-> billing;
	classify -.-> general;
	classify -.-> human;
	classify -.-> technical;
	billing --> __end__;
	general --> __end__;
	human --> __end__;
	technical --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [8]:
examples = [
    "Tôi bị tính phí hai lần trong tháng này.",
    "Ứng dụng báo lỗi 401 sau khi tôi đổi API key.",
    "Bên bạn làm việc từ mấy giờ?",
]

for question in examples:
    result = support_graph.invoke({"question": question})
    print("\nCâu hỏi:", question)
    print("Route:", result["category"], "| confidence:", result["confidence"])
    print("Lý do:", result["rationale"])
    print("Trả lời:", result["answer"])


Câu hỏi: Tôi bị tính phí hai lần trong tháng này.
Route: billing | confidence: 0.95
Lý do: Yêu cầu liên quan đến việc bị tính phí hai lần, thuộc vấn đề thanh toán/hóa đơn.
Trả lời: Chào bạn,

Để xác minh nguyên nhân tính phí hai lần trong tháng này, mình cần một số thông tin cơ bản của tài khoản bạn. Bạn vui lòng cung cấp:

1. **Tên đăng nhập / Email** (được dùng khi đăng ký dịch vụ)  
2. **Số hóa đơn hoặc mã giao dịch** (nếu bạn đã nhận được)  
3. **Ngày và thời gian giao dịch** (khoảng thời gian bạn nhận được khoản phí thứ hai)

Nếu bạn không có thông tin này, mình sẽ hướng dẫn cách kiểm tra lịch sử giao dịch trong tài khoản của bạn hoặc liên hệ với bộ phận hỗ trợ khách hàng để lấy dữ liệu chi tiết.

Lưu ý: mình không thể tạo hoặc giả mạo dữ liệu tài khoản. Nếu thiếu dữ liệu, mình sẽ thông báo rõ ràng và chỉ đưa ra các bước tiếp theo dựa trên thông tin bạn cung cấp.

Câu hỏi: Ứng dụng báo lỗi 401 sau khi tôi đổi API key.
Route: technical | confidence: 0.95
Lý do: Lỗi 401 khi đổi API

## Bài tập trên lớp

1. Thêm category `sales` và node tư vấn bán hàng.
2. Đổi threshold từ `0.55` lên `0.8`; đo số yêu cầu rơi vào `human`.
3. Thêm field `trace` để ghi lại các node đã chạy.

**Gợi ý vận hành:** model trên Groq có thể thay đổi theo tài khoản/thời điểm. Nếu model mặc định không khả dụng, đặt biến môi trường `GROQ_MODEL` thành model mà API key của bạn được phép dùng trong Groq Console rồi chạy lại.